In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from modelscope import snapshot_download

/root/nour/nourenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [11]:
model_id = "Qwen/Qwen3-1.7B" # Or 0.6B for speed
model_dir = snapshot_download(model_id , cache_dir="./models")

2026-03-25 21:07:34,099 - modelscope - INFO - Got 12 files, start to download ...
Processing 12 items:   0%|          | 0.00/12.0 [00:00<?, ?it/s]




























Processing 12 items:   8%|▊         | 1.00/12.0 [00:00<00:04, 2.52it/s]





























Processing 12 items:  58%|█████▊    | 7.00/12.0 [00:00<00:00, 10.6it/s]














































































































































































































































































































































































































































































































































































































































































In [8]:
local_model_path = "models/Qwen/Qwen3-1.7B" # Path to the downloaded model directory

# 1. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    local_model_path, 
    local_files_only=True,
    trust_remote_code=True
)


# Load model in 4-bit to save memory (important for keyboard research)
model = AutoModelForCausalLM.from_pretrained(
    local_model_path, 
    device_map=device, 
    torch_dtype=torch.float16,
    #load_in_4bit=True,
    local_files_only=True,
    trust_remote_code=True,
)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 431.01it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [9]:
def predict_next(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=True, top_k=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# TEST YOUR THREE LANGUAGES
print("EN:", predict_next("How are you"))
print("FR:", predict_next("Comment vas-"))
print("ZH:", predict_next("Ni hao ma (你好吗)")) 
print("MIX:", predict_next("I want to eat some 饺子 because it is very"))

EN: How are you managing the time between your
FR: Comment vas-je savoir si mon enf
ZH: Ni hao ma (你好吗) 翻译成
MIX: I want to eat some 饺子 because it is very delicious, but I feel
